| Mục | Chi tiết cụ thể |
|:---|:---|
| **Dữ liệu đầu vào** | `movies.dat` và `ratings.dat` |
| **Chia dữ liệu** | Random shuffle → 30% train, 20% validation, 50% hidden test |
| **User-based Collaborative Filtering** | - Tính cosine similarity giữa các user<br>- Dự đoán rating bằng cách weighted average các user tương tự<br>- Không lọc top-k users (lấy tất cả các user có rating) |
| **Item-based Collaborative Filtering** | - Tính cosine similarity giữa các item (movies)<br>- Dự đoán rating bằng weighted average các item tương tự<br>- Cũng không lọc top-k items (lấy tất cả phim user đã xem) |
| **Đánh giá RMSE** | - CF (User-based và Item-based): dùng `sqrt(mean_squared_error)`<br>- SVD: dùng `surprise.accuracy.rmse` |


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise.model_selection import GridSearchCV
from surprise import accuracy
from sklearn.metrics.pairwise import cosine_similarity
from math import sqrt
from sklearn.metrics import mean_squared_error



In [ ]:
#đổi đường dẫn này theo thư mục của bạn
movies = pd.read_csv("/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/1m/movies.dat", sep="::", engine="python", 
                     names=["movieId", "title", "genres"], encoding="latin1")

# Đọc ratings.dat và đổi tên cột MovieID -> ItemID
ratings = pd.read_csv("/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/1m/ratings.dat", sep="::", engine="python", 
                      names=["userId", "movieId", "rating", "timestamp"])

In [3]:
print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
   movieId                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy


In [4]:
# Xóa cột timestamp nếu không cần thiết
ratings = ratings.drop(columns=["timestamp"])

# Chuyển về định dạng phù hợp cho thư viện Surprise
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)


In [5]:
# 2. Chia dữ liệu
ratings_shuffled = ratings.sample(frac=1, random_state=42).reset_index(drop=True)
n_total = len(ratings_shuffled)
n_train = int(0.3 * n_total)
n_valid = int(0.2 * n_total)

train_data = ratings_shuffled[:n_train]
valid_data = ratings_shuffled[n_train:n_train + n_valid]
hidden_test_data = ratings_shuffled[n_train + n_valid:]

In [6]:
# 3. User-Item matrix
user_item_matrix = train_data.pivot(index='userId', columns='movieId', values='rating')
user_item_matrix_filled = user_item_matrix.fillna(0)

# 4. Tính Similarity
user_similarity = cosine_similarity(user_item_matrix_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

item_similarity = cosine_similarity(user_item_matrix_filled.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)


In [7]:
# 5. Hàm dự đoán

def predict_user_based(user_id, movie_id):
    if movie_id not in user_item_matrix.columns or user_id not in user_item_matrix.index:
        return np.nan

    sim_users = user_similarity_df[user_id]
    movie_ratings = user_item_matrix[movie_id]
    mask = movie_ratings.notna()

    if mask.sum() == 0:
        return np.nan

    sim_scores = sim_users[mask]
    ratings = movie_ratings[mask]

    if sim_scores.sum() == 0:
        return np.nan

    prediction = np.dot(sim_scores, ratings) / sim_scores.sum()
    return prediction

def predict_item_based(user_id, movie_id):
    if user_id not in user_item_matrix.index:
        return np.nan

    user_ratings = user_item_matrix.loc[user_id]
    mask = user_ratings.notna()

    if mask.sum() == 0:
        return np.nan

    similar_items = item_similarity_df.get(movie_id)
    if similar_items is None:
        return np.nan

    similar_items = similar_items[mask]

    if similar_items.sum() == 0:
        return np.nan

    ratings = user_ratings[mask]
    prediction = np.dot(similar_items, ratings) / similar_items.sum()
    return prediction

In [8]:
# 7. Đánh giá

def evaluate_cf(predict_function, data):
    y_true = []
    y_pred = []

    for _, row in data.iterrows():
        uid = row['userId']
        iid = row['movieId']
        true_rating = row['rating']
        pred_rating = predict_function(uid, iid)

        if np.isnan(pred_rating):
            continue
        
        y_true.append(true_rating)
        y_pred.append(pred_rating)

    rmse = sqrt(mean_squared_error(y_true, y_pred))
    return rmse



In [9]:
# 8. Chạy đánh giá

print("=== Đánh giá trên Validation Set ===")
rmse_user_valid = evaluate_cf(predict_user_based, valid_data)
rmse_item_valid = evaluate_cf(predict_item_based, valid_data)


print(f"User-Based CF Validation RMSE: {rmse_user_valid:.4f}")
print(f"Item-Based CF Validation RMSE: {rmse_item_valid:.4f}")


print("\n=== Đánh giá trên Hidden Test Set ===")
rmse_user_test = evaluate_cf(predict_user_based, hidden_test_data)
rmse_item_test = evaluate_cf(predict_item_based, hidden_test_data)


print(f"User-Based CF Test RMSE: {rmse_user_test:.4f}")
print(f"Item-Based CF Test RMSE: {rmse_item_test:.4f}")


=== Đánh giá trên Validation Set ===
User-Based CF Validation RMSE: 0.9776
Item-Based CF Validation RMSE: 1.0112

=== Đánh giá trên Hidden Test Set ===
User-Based CF Test RMSE: 0.9790
Item-Based CF Test RMSE: 1.0123


In [1]:
# --- Import libraries ---
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

# --- Load your data ---
ratings_data = pd.read_csv("/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/1m/ratings.dat", sep='::', #đổi đường dẫn này theo thư mục của bạn
                            names=['user_id', 'movie_id', 'rating', 'timestamp'], engine='python')

# --- Pivot ratings matrix ---
ratings_matrix = ratings_data.pivot(index='user_id', columns='movie_id', values='rating')

# --- Fill NA with 0 (for similarity calc) ---
rating_matrix_filled = ratings_matrix.fillna(0)

# --- Compute similarity matrices ---
user_similarity = cosine_similarity(rating_matrix_filled)
item_similarity = cosine_similarity(rating_matrix_filled.T)

user_similarity_df = pd.DataFrame(user_similarity, index=rating_matrix_filled.index, columns=rating_matrix_filled.index)
item_similarity_df = pd.DataFrame(item_similarity, index=rating_matrix_filled.columns, columns=rating_matrix_filled.columns)

# --- Split into train, val, test ---
# Split dataset theo tỷ lệ 80% Train – 10% Val – 10% Test
train_data, temp_data = train_test_split(ratings_data, test_size=0.2, random_state=42)         # 80% train, 20% temp
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)              # temp 20% tách tiếp 10% val + 10% test

# --- Pivot full matrix from train only ---
ratings_matrix = train_data.pivot(index='user_id', columns='movie_id', values='rating')
rating_matrix_filled = ratings_matrix.fillna(0)

# --- Recompute similarities using train set ---
user_similarity = cosine_similarity(rating_matrix_filled)
item_similarity = cosine_similarity(rating_matrix_filled.T)

user_similarity_df = pd.DataFrame(user_similarity, index=rating_matrix_filled.index, columns=rating_matrix_filled.index)
item_similarity_df = pd.DataFrame(item_similarity, index=rating_matrix_filled.columns, columns=rating_matrix_filled.columns)

# --- Predict function ---
def predict_user_cf(user_id, item_id, k=20):
    if user_id not in user_similarity_df.index or item_id not in ratings_matrix.columns:
        return np.nan
    sim_users = user_similarity_df[user_id].drop(index=user_id).nlargest(k)
    ratings = ratings_matrix.loc[sim_users.index, item_id]
    weighted_ratings = sim_users * ratings.fillna(0)
    return weighted_ratings.sum() / (np.abs(sim_users[ratings.notna()]).sum() + 1e-8)

def predict_item_cf(user_id, item_id, k=20):
    if item_id not in item_similarity_df.index or user_id not in ratings_matrix.index:
        return np.nan
    sim_items = item_similarity_df[item_id].drop(index=item_id).nlargest(k)
    ratings = ratings_matrix.loc[user_id, sim_items.index]
    weighted_ratings = sim_items * ratings.fillna(0)
    return weighted_ratings.sum() / (np.abs(sim_items[ratings.notna()]).sum() + 1e-8)

# --- Apply prediction on test set ---
test_data = test_data.copy()
test_data['pred_user_cf'] = test_data.apply(lambda row: predict_user_cf(row['user_id'], row['movie_id']), axis=1)
test_data['pred_item_cf'] = test_data.apply(lambda row: predict_item_cf(row['user_id'], row['movie_id']), axis=1)

# --- Drop NA predictions ---
test_data = test_data.dropna(subset=['pred_user_cf', 'pred_item_cf'])

# --- Evaluate ---
# Tính MAE và RMSE
mae_user = mean_absolute_error(test_data['rating'], test_data['pred_user_cf'])
rmse_user = np.sqrt(mean_squared_error(test_data['rating'], test_data['pred_user_cf']))

mae_item = mean_absolute_error(test_data['rating'], test_data['pred_item_cf'])
rmse_item = np.sqrt(mean_squared_error(test_data['rating'], test_data['pred_item_cf']))

# Hiển thị kết quả
print(f"🔹 User-based CF MAE: {mae_user:.4f}, RMSE: {rmse_user:.4f}")
print(f"🔹 Item-based CF MAE: {mae_item:.4f}, RMSE: {rmse_item:.4f}")


🔹 User-based CF MAE: 0.9429, RMSE: 1.2962
🔹 Item-based CF MAE: 0.8343, RMSE: 1.1751
